# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and is accessible at:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# The metadata object from mlcroissant is an object, not a dict
metadata = dataset.metadata

# Print summary information
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their @id
print("Available record sets and their @id:")
record_sets = getattr(metadata, 'recordSet', [])
record_set_ids = []
for rs in record_sets:
    print(f"- {getattr(rs, '@id', getattr(rs, 'id', None))}: {getattr(rs, 'name', 'No name')}")
    record_set_ids.append(getattr(rs, '@id', getattr(rs, 'id', None)))
if not record_set_ids:
    print("No record sets explicitly listed in the metadata.")

In [ ]:
# If record sets are available, print sample records and field @id; else, try listing columns from a fallback distribution
for rs in getattr(metadata, 'recordSet', []):
    print(f"\nRecord set {getattr(rs, '@id', getattr(rs, 'id', None))} ({getattr(rs, 'name', 'No name')}):")
    print(f"Fields:")
    for field in getattr(rs, 'field', []):
        print(f"  - {getattr(field, '@id', getattr(field, 'id', None))}: {getattr(field, 'name', 'No name')} ({getattr(field, 'dataType', 'No dataType')})")
    # Preview the first 1-2 records
    try:
        records = list(dataset.records(record_set=getattr(rs, '@id', getattr(rs, 'id', None))))[:2]
        for rec in records:
            pprint.pprint(rec)
    except Exception as e:
        print(f"Could not preview records: {e}")

# If not, print a note for the user to check available columns in next step.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
All references use the entity `@id` as per Croissant schema.

In [ ]:
# Prepare to extract data from available record sets
# If no recordSet entities explicitly, fall back to the first distribution
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
else:
    # Fallback: get contentUrl from distribution for direct loading
    distrs = getattr(metadata, 'distribution', [])
    chosen_record_set_id = None
    if distrs:
        # Use the first distribution's @id
        chosen_record_set_id = getattr(distrs[0], '@id', getattr(distrs[0], 'id', None))

# Extract into DataFrame
dataframes = {}
if chosen_record_set_id:
    try:
        records = list(dataset.records(record_set=chosen_record_set_id))
        df = pd.DataFrame(records)
        dataframes[chosen_record_set_id] = df
        print(f"Columns for record set {chosen_record_set_id}:")
        print(df.columns.tolist())
        print("Preview:")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {chosen_record_set_id}: {e}")
else:
    print("No record set or distribution available to load.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping by attributes.
All fields are referenced by their `@id` where available.

In [ ]:
# Select a numeric field for analysis via its @id
numeric_field_id = None
group_field_id = None

# If recordSet fields info was available, identify a numeric field such as GAD-7, PHQ-9, or PSQ score
if chosen_record_set_id and chosen_record_set_id in dataframes and not dataframes[chosen_record_set_id].empty:
    df = dataframes[chosen_record_set_id]
    # Try to auto-select a numeric column
    for col in df.columns:
        if any(score in col.lower() for score in ['gad', 'phq', 'psq']) and df[col].dtype in [int, float] or df[col].dtype == 'object' and df[col].str.isnumeric().any():
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Try to auto-select any int/float column
        for col in df.columns:
            if df[col].dtype in [int, float]:
                numeric_field_id = col
                break

    group_field_id = None
    for col in df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break
        if 'education' in col.lower():
            group_field_id = col
            break

    if numeric_field_id:
        # Optional: Convert numeric column to numeric if not already
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please check columns above.")
else:
    print("No record set DataFrame available.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib and pandas plotting tools.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if chosen_record_set_id and numeric_field_id and chosen_record_set_id in dataframes:
    df = dataframes[chosen_record_set_id]
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is available, plot group means
    if group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        plt.figure(figsize=(6,4))
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric field or data available for visualization.")

## 6. Conclusion
Through this notebook, we've loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.
- The dataset structure and fields are discoverable via their `@id` referencing according to the Croissant schema.
- We performed basic EDA, including filtering and normalization, and visualized numeric data (e.g., GAD-7, PHQ-9, or PSQ scores).
- The dataset enables further analysis to understand mental health trends and community health dynamics in Kilifi County.

For more complex analyses, please refer to the official `mlcroissant` documentation or extend this notebook to include additional visualizations and processing.